# 닥터그린 딸기 병해 YOLO — mAP 향상 학습 노트북 (Colab Pro)

이 노트북은 Hugging Face Space `henna22/doctor-green-strawberry`에 배포된 딸기 병해 YOLO 탐지 모델의
**mAP를 측정하고, 데이터를 감사하고, 재학습해서, 안전하게 재배포**하기 위한 전체 파이프라인입니다.

**전체 흐름**

```
[0 설정] → [1 환경 준비] → [2 기존 모델 가져오기(버전 자동 감지)]
       → [3 데이터 준비(라벨 변환 + 층화 분할)] → [4 베이스라인 측정]
       → [5 데이터 감사] → [6 재학습 (프리셋 A/B/C)]
       → [7 비교 평가] → [8 오류 분석] → [9 배포]
```

**사용법**

1. 코랩 메뉴에서 **런타임 → 런타임 유형 변경 → GPU (L4 또는 A100)** 를 선택하세요.
2. 아래 **0. 설정(CONFIG)** 셀의 값만 본인 환경에 맞게 고치세요. 나머지 셀은 수정 없이 위에서 아래로 순서대로 실행합니다.
3. 학습 결과·CSV·가중치는 전부 Google Drive(`PROJECT_DIR`)에 저장되므로 **세션이 끊겨도 보존**됩니다.

**앱 연동 시 반드시 알아야 할 제약**

- 닥터그린 앱은 **신뢰도 컷오프 0.75**를 사용합니다(`app/diagnose/result/lib.ts`). conf 0.75 미만 탐지는 앱에서 버려집니다.
- Space의 Gradio 서버(app.py)가 모델 출력을 `disease_name`, `confidence`, `detections[].name/box` 스키마로 후처리합니다.
  → **클래스 이름과 순서가 바뀌면 앱이 깨집니다.** 9장의 검증 셀을 통과한 가중치만 업로드하세요.

## 0. 설정 (CONFIG)

사용자가 고칠 값은 **이 셀 하나**에 모두 모여 있습니다. 아래 셀들은 전부 이 값을 참조합니다.

In [ ]:
# ======================= 사용자 설정 — 여기만 고치세요 =======================

# prep_win.py가 만든 dataset 폴더(images/ + labels/ + data.yaml)를 통째로 Google Drive에 올린 위치
DATASET_DIR = '/content/drive/MyDrive/doctor_green_dataset'

# 라벨 형식: prep_win.py 출력은 이미 YOLO txt이므로 'yolo' 고정 (바꾸지 마세요)
LABEL_FORMAT = 'yolo'

# 이미 완성된 data.yaml이 있으면 경로 지정(층화 분할을 건너뜀). 없으면 ''(자동 생성, 권장)
DATA_YAML = ''

# COCO 형식일 때만: 통합 annotation JSON 경로. ''이면 DATASET_DIR에서 자동 탐색
COCO_JSON = ''

# 기존 배포 모델이 있는 Hugging Face Space
HF_SPACE = 'henna22/doctor-green-strawberry'

# 기존 가중치(.pt) 경로. ''이면 위 Space에서 자동 다운로드
BASELINE_WEIGHTS = ''

# 모든 산출물(학습 결과, CSV, best.pt)이 저장될 Drive 폴더 — 세션이 끊겨도 보존됨
PROJECT_DIR = '/content/drive/MyDrive/doctor_green_training'

# 학습 프리셋: 'A' 빠른 개선(~1시간) | 'B' 정확도 우선(2~4시간) | 'C' 하이퍼파라미터 탐색(반나절)
PRESET = 'A'

# 랜덤 시드 (층화 분할·학습 재현용 — 바꾸지 않으면 세션이 끊겨도 같은 분할이 재현됨)
SEED = 42

# ---- AI Hub JSON 라벨일 때만: 키 이름 매핑 (데이터셋마다 다르므로 실제 JSON을 열어 확인 후 조정) ----
AIHUB_KEYS = {
    'annotations': 'annotations',   # 객체(병반) 목록이 들어 있는 키
    'class':       'disease',       # 질병(클래스) 이름 키 — 객체 안 또는 최상위에서 찾음
    'bbox':        'points',        # 바운딩박스 키
    'width':       'width',         # 이미지 가로 크기 키 (없으면 이미지 파일에서 읽음)
    'height':      'height',        # 이미지 세로 크기 키
}
# bbox 값의 형태: 'xyxy_dict'({'xtl','ytl','xbr','ybr'}) | 'xyxy'([x1,y1,x2,y2]) | 'xywh'([x,y,w,h])
AIHUB_BBOX_MODE = 'xyxy_dict'

# ======================= 이 아래는 고치지 않아도 됩니다 =======================
import os

def need(*names):
    # 이전 셀에서 만들어졌어야 할 변수가 없으면 한국어로 안내하고 셀 실행을 건너뛰게 하는 가드
    missing = [n for n in names if n not in globals()]
    if missing:
        print('[중단] 아직 준비되지 않은 값이 있습니다:', ', '.join(missing))
        print('       위쪽 셀(특히 1~3장)을 순서대로 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    return True


# 개체ID(누수 방지 그룹) 정규식 — 3-2 층화 분할에서 같은 개체(같은 딸기 포기)를 연속
# 촬영한 프레임이 train/val/test에 나뉘어 들어가는 걸 막는 데 씀. 비우면 파일명 끝의
# 프레임/일련번호(타임스탬프 등, 연속된 숫자)를 자동으로 떼어 개체 키를 추정한다.
# AI Hub 딸기 데이터 파일명(예: 딸기_설향_황화_23_006_220924173503)이면 기본값(빈 문자열)으로 충분.
GROUP_ID_REGEX = ''

print('CONFIG 로드 완료')
print('  DATASET_DIR :', DATASET_DIR)
print('  LABEL_FORMAT:', LABEL_FORMAT, '| PRESET:', PRESET, '| SEED:', SEED)
print('  PROJECT_DIR :', PROJECT_DIR)

## 1. 환경 준비

ultralytics(YOLO 통합 인터페이스)와 huggingface_hub를 설치하고, GPU와 Drive를 확인합니다.
GPU가 없으면 학습이 수십 배 느려지므로 여기서 반드시 확인하세요.

In [ ]:
# ultralytics/huggingface_hub 버전을 고정합니다(최신값으로 자동 갱신하지 않음).
# API 변경으로 tune()/val() 등이 깨지는 것을 방지하기 위함 — 필요 시 버전을 올리고
# 이 노트북(3~9장)을 처음부터 다시 검증한 뒤에만 실제 학습/배포에 사용하세요.
!pip install -q ultralytics==8.4.95 huggingface_hub==1.23.0
!nvidia-smi

import torch
import ultralytics

print()
print('ultralytics 버전:', ultralytics.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('[경고] GPU가 감지되지 않았습니다!')
    print('       코랩 메뉴: 런타임 → 런타임 유형 변경 → GPU(L4/A100) 선택 후 다시 실행하세요.')

# Google Drive 마운트 (데이터셋 읽기 + 결과 보존용)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('[안내] Drive 마운트 실패(코랩 환경이 아닐 수 있음):', e)

try:
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print('결과 저장 폴더 준비 완료:', PROJECT_DIR)
except NameError:
    print('[중단] 0장(CONFIG) 셀을 먼저 실행하세요.')

## 2. 기존 모델 가져오기 + 버전 자동 감지

배포 중인 Space에서 `.pt` 가중치를 자동으로 찾아 내려받고, 체크포인트를 열어
**YOLO 버전(v5/v8/11 등)과 클래스 이름 목록**을 확인합니다.
클래스 이름은 3장(데이터 준비)과 9장(배포 검증)에서 기준값으로 재사용됩니다.

> Space가 private이면 먼저 `from huggingface_hub import notebook_login; notebook_login()`을 실행해
> read 권한 토큰으로 로그인한 뒤 이 셀을 다시 실행하세요.

In [ ]:
import glob
from pathlib import Path

baseline_weights_path = None
space_pt_relpath = None  # Space 안에서의 상대 경로 — 9장 업로드 때 같은 자리에 교체하기 위해 기억

try:
    if BASELINE_WEIGHTS:
        if os.path.exists(BASELINE_WEIGHTS):
            baseline_weights_path = BASELINE_WEIGHTS
            print('CONFIG에 지정된 가중치를 사용합니다:', baseline_weights_path)
        else:
            print('[오류] BASELINE_WEIGHTS 경로에 파일이 없습니다:', BASELINE_WEIGHTS)
            print('       Drive 마운트 여부와 경로 오타를 확인하세요.')
    else:
        from huggingface_hub import snapshot_download
        print(f'Hugging Face Space에서 다운로드 중: {HF_SPACE} ...')
        space_dir = snapshot_download(repo_id=HF_SPACE, repo_type='space', local_dir='/content/hf_space')
        pt_files = sorted(Path(space_dir).rglob('*.pt'), key=lambda p: p.stat().st_size, reverse=True)
        if not pt_files:
            print('[오류] Space 안에서 .pt 파일을 찾지 못했습니다.')
            print('       가중치가 별도 모델 저장소나 외부 URL에 있을 수 있습니다.')
            print('       해결: Space의 app.py에서 가중치 로드 부분을 확인하고,')
            print('             파일을 Drive에 올린 뒤 CONFIG의 BASELINE_WEIGHTS에 경로를 넣으세요.')
        else:
            print(f'.pt 파일 {len(pt_files)}개 발견:')
            for p in pt_files:
                print(f'  - {p.relative_to(space_dir)} ({p.stat().st_size/1e6:.1f} MB)')
            baseline_weights_path = str(pt_files[0])
            space_pt_relpath = str(pt_files[0].relative_to(space_dir))
            print()
            print('가장 큰 파일을 베이스라인으로 사용합니다:', space_pt_relpath)
except Exception as e:
    print('[오류] Space 다운로드 실패:', e)
    print('힌트 1) 401/403 → Space가 private입니다. notebook_login() 후 재실행하세요.')
    print('힌트 2) 네트워크 오류 → 셀을 다시 실행해 보세요.')
    print('힌트 3) 계속 실패하면 가중치를 Drive에 올리고 BASELINE_WEIGHTS에 경로를 넣으세요.')

In [ ]:
# 체크포인트를 열어 YOLO 버전·클래스 이름을 자동 감지합니다.
from ultralytics import YOLO

baseline_model = None
baseline_names = {}
BASE_IMGSZ = 640

if need('baseline_weights_path') and baseline_weights_path:
    try:
        baseline_model = YOLO(baseline_weights_path)
        baseline_names = dict(baseline_model.names)
        print('모델 로드 성공. 클래스 수:', len(baseline_names))
        print('클래스 이름(이 순서가 앱 스키마의 기준입니다):')
        for i in sorted(baseline_names):
            print(f'  {i}: {baseline_names[i]}')
        print()
        baseline_model.info()
    except Exception as e:
        print('[오류] ultralytics YOLO 클래스로 로드 실패:', e)
        print('힌트 1) 원조 YOLOv5 저장소(2020~2022) 체크포인트는 ultralytics 패키지로 직접 로드가 안 될 수 있습니다.')
        print('        → 코랩에서 git clone https://github.com/ultralytics/yolov5 후 export.py로 재저장하거나,')
        print('          이 노트북 6장에서 새로 학습한 가중치로 교체하는 것을 권장합니다.')
        print('힌트 2) 파일이 손상됐을 수 있습니다. 다운로드를 다시 시도하세요.')

    # 버전·학습 당시 설정 추정 (실패해도 치명적이지 않으므로 별도 try)
    try:
        import torch
        ckpt = torch.load(baseline_weights_path, map_location='cpu', weights_only=False)
        train_args = ckpt.get('train_args') or {}
        yaml_file = ''
        if ckpt.get('model') is not None and hasattr(ckpt['model'], 'yaml'):
            yaml_file = str(ckpt['model'].yaml.get('yaml_file', ''))
        clue = (yaml_file + ' ' + str(train_args.get('model', '')) + ' ' + os.path.basename(baseline_weights_path)).lower()
        version = '알 수 없음'
        for tag, label in [('yolov5', 'YOLOv5 (ultralytics 포맷)'), ('yolov8', 'YOLOv8'),
                           ('yolov9', 'YOLOv9'), ('yolov10', 'YOLOv10'),
                           ('yolo11', 'YOLO11'), ('yolo12', 'YOLO12')]:
            if tag in clue:
                version = label
                break
        BASE_IMGSZ = int(train_args.get('imgsz', 640) or 640)
        print()
        print('=== 자동 감지 결과 ===')
        print('  추정 YOLO 버전       :', version, f'(단서: "{clue.strip()}")')
        print('  저장 당시 ultralytics:', ckpt.get('version', '기록 없음'))
        print('  학습 당시 imgsz      :', BASE_IMGSZ, '(베이스라인 평가에 그대로 사용)')
        print('  학습 당시 epochs     :', train_args.get('epochs', '기록 없음'))
    except Exception as e:
        print('[안내] 체크포인트 메타데이터를 읽지 못했습니다(평가는 계속 가능):', e)

## 3. 데이터 준비 — 라벨 변환 + 층화 분할

`LABEL_FORMAT`에 따라 라벨을 YOLO txt로 통일한 뒤, **클래스별 비율을 유지하는 층화 분할**로
train/val/test = 8:1:1을 만들고 `data.yaml`을 자동 생성합니다.
분할이 무작위·불균형이면 mAP 수치 자체를 믿을 수 없게 되므로, 이 단계가 모든 측정의 기초입니다.

> **데이터 누수 경고 — mAP가 뻥튀기되는 가장 흔한 원인**
> 같은 개체(같은 딸기 포기·같은 병반)를 연속 촬영한 사진이 train과 val/test에 나뉘어 들어가면,
> 모델이 "본 적 있는 장면"으로 평가받아 mAP가 실제보다 크게 부풀려집니다.
> 이 노트북의 재분할은 이제 파일명 기반 그룹 분할로 연속 프레임 누수를 방지합니다
> (완벽하진 않음 — 파일명 규칙이 다르면 GROUP_ID_REGEX 조정). 5장의 중복(해시) 검사는
> 완전 동일 이미지 누수를 추가로 잡아줍니다.

> 분할 결과는 세션 로컬(`/content/dataset`)에 만들어 학습 속도를 높입니다.
> **세션이 끊기면 이 장(3장)의 두 셀만 다시 실행**하면 같은 SEED로 동일한 분할이 재현됩니다.

In [ ]:
# 3-1) 이미지 수집 + 라벨을 YOLO txt로 통일
import json as _json
import random
import shutil
from pathlib import Path
from PIL import Image

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_images(root):
    return sorted(p for p in Path(root).rglob('*') if p.suffix.lower() in IMG_EXTS)

def read_yaml_names(root):
    # 데이터셋 폴더 안에 이미 data.yaml류가 있으면 클래스 이름을 빌려옴
    import yaml
    for yp in sorted(Path(root).rglob('*.yaml')):
        try:
            d = yaml.safe_load(open(yp, encoding='utf-8'))
            if isinstance(d, dict) and 'names' in d:
                n = d['names']
                names = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
                print(f'기존 yaml에서 클래스 이름을 가져왔습니다: {yp}')
                return names
        except Exception:
            pass
    return None

def base_name_to_id():
    # 배포 모델의 클래스 순서를 최우선으로 사용 → 앱 스키마 유지
    if 'baseline_names' in globals() and baseline_names:
        return {v: k for k, v in baseline_names.items()}
    return {}

pairs = []        # (이미지 경로, YOLO txt 라벨 경로 또는 None)
class_names = []  # 인덱스 순서 = 클래스 id 순서

images = find_images(DATASET_DIR)
print(f'{DATASET_DIR} 아래에서 이미지 {len(images)}장 발견')
if not images:
    print('[오류] 이미지가 없습니다. DATASET_DIR 경로와 Drive 마운트를 확인하세요.')

CONV_DIR = Path('/content/converted_labels')

if images and LABEL_FORMAT == 'yolo':
    # (a) 이미 YOLO txt → 폴더 구조 검증만
    def find_label(img):
        cands = [img.with_suffix('.txt')]
        s = str(img)
        if '/images/' in s:
            cands.append(Path(s.replace('/images/', '/labels/')).with_suffix('.txt'))
        for c in cands:
            if c.exists():
                return c
        return None

    max_id = -1
    for img in images:
        lbl = find_label(img)
        pairs.append((img, lbl))
        if lbl is not None:
            try:
                for line in open(lbl, encoding='utf-8'):
                    if line.strip():
                        max_id = max(max_id, int(float(line.split()[0])))
            except Exception:
                pass

    n_missing = sum(1 for _, l in pairs if l is None)
    print(f'라벨 매칭: {len(pairs) - n_missing}/{len(pairs)}장 (라벨 없음 {n_missing}장 → 배경 이미지로 취급)')
    print(f'라벨에 등장한 최대 클래스 id: {max_id}')

    if baseline_names and max_id < len(baseline_names):
        class_names = [baseline_names[i] for i in sorted(baseline_names)]
        print('클래스 이름: 배포 모델(baseline)의 이름/순서를 그대로 사용합니다. (권장)')
    else:
        found = read_yaml_names(DATASET_DIR)
        if found and max_id < len(found):
            class_names = found
        else:
            class_names = [f'class_{i}' for i in range(max_id + 1)]
            print('[주의] 클래스 이름을 찾지 못해 임시 이름(class_0...)을 씁니다.')
            print('       배포 전 9장에서 반드시 실제 질병 이름으로 맞춰야 합니다.')

elif images and LABEL_FORMAT == 'aihub_json':
    # (b) AI Hub식 이미지별 JSON → YOLO txt 변환 (키 이름은 CONFIG의 AIHUB_KEYS로 조정)
    if CONV_DIR.exists():
        shutil.rmtree(CONV_DIR)
    CONV_DIR.mkdir(parents=True)
    json_map = {p.stem: p for p in Path(DATASET_DIR).rglob('*.json')}
    print(f'JSON 라벨 {len(json_map)}개 발견')

    name_to_id = base_name_to_id()
    if name_to_id:
        print('클래스 id는 배포 모델의 이름/순서에 맞춥니다. (앱 스키마 유지)')
    new_names = []          # 배포 모델에 없던 새 클래스
    shown_key_error = False
    n_ok, n_skip = 0, 0

    def parse_bbox(bb):
        if AIHUB_BBOX_MODE == 'xyxy_dict':
            for keys in (('xtl', 'ytl', 'xbr', 'ybr'), ('x1', 'y1', 'x2', 'y2')):
                if all(k in bb for k in keys):
                    return [float(bb[k]) for k in keys]
            return None
        vals = [float(v) for v in bb]
        if AIHUB_BBOX_MODE == 'xyxy':
            return vals[:4]
        if AIHUB_BBOX_MODE == 'xywh':
            x, y, w, h = vals[:4]
            return [x, y, x + w, y + h]
        return None

    for img in images:
        jp = json_map.get(img.stem)
        if jp is None:
            pairs.append((img, None))
            continue
        try:
            data = _json.load(open(jp, encoding='utf-8'))
        except Exception:
            n_skip += 1
            pairs.append((img, None))
            continue
        anns = data.get(AIHUB_KEYS['annotations'])
        if anns is None:
            if not shown_key_error:
                shown_key_error = True
                print(f"[오류] JSON에 '{AIHUB_KEYS['annotations']}' 키가 없습니다. 실제 최상위 키: {list(data.keys())}")
                print('       CONFIG의 AIHUB_KEYS를 실제 키 이름으로 고치고 이 셀을 다시 실행하세요.')
            n_skip += 1
            pairs.append((img, None))
            continue
        if isinstance(anns, dict):
            anns = [anns]
        W = data.get(AIHUB_KEYS['width'])
        H = data.get(AIHUB_KEYS['height'])
        if not W or not H:
            with Image.open(img) as im:
                W, H = im.size
        W, H = float(W), float(H)
        lines = []
        for ann in anns:
            if not isinstance(ann, dict):
                continue
            cls_val = ann.get(AIHUB_KEYS['class'], data.get(AIHUB_KEYS['class']))
            bb = ann.get(AIHUB_KEYS['bbox'])
            if cls_val is None or bb is None:
                continue
            xyxy = parse_bbox(bb)
            if xyxy is None:
                continue
            x1, y1, x2, y2 = xyxy
            x1, x2 = max(0.0, min(x1, x2)), min(W, max(x1, x2))
            y1, y2 = max(0.0, min(y1, y2)), min(H, max(y1, y2))
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            cls_val = str(cls_val)
            if cls_val not in name_to_id:
                name_to_id[cls_val] = len(name_to_id)
                new_names.append(cls_val)
            cid = name_to_id[cls_val]
            cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
            bw, bh = (x2 - x1) / W, (y2 - y1) / H
            lines.append(f'{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        out = CONV_DIR / (img.stem + '.txt')
        out.write_text('\n'.join(lines), encoding='utf-8')
        pairs.append((img, out))
        n_ok += 1

    class_names = [None] * len(name_to_id)
    for n, i in name_to_id.items():
        class_names[i] = n
    print(f'변환 완료: {n_ok}장 성공, {n_skip}장 건너뜀')
    print('클래스 목록:', class_names)
    if new_names and baseline_names:
        print(f'[경고] 배포 모델에 없던 새 클래스 {len(new_names)}개가 발견됐습니다: {new_names}')
        print('       이대로 학습해 배포하면 Space app.py의 disease_name 매핑이 깨집니다. 9장 검증 필수!')

elif images and LABEL_FORMAT == 'coco':
    # (c) COCO 통합 JSON → YOLO txt 변환
    coco_path = COCO_JSON
    if not coco_path:
        for jp in sorted(Path(DATASET_DIR).rglob('*.json')):
            try:
                d = _json.load(open(jp, encoding='utf-8'))
                if isinstance(d, dict) and 'images' in d and 'annotations' in d:
                    coco_path = str(jp)
                    break
            except Exception:
                pass
    if not coco_path:
        print('[오류] COCO annotation JSON을 찾지 못했습니다. CONFIG의 COCO_JSON에 경로를 지정하세요.')
    else:
        print('COCO JSON 사용:', coco_path)
        coco = _json.load(open(coco_path, encoding='utf-8'))
        cats = sorted(coco['categories'], key=lambda c: c['id'])
        name_to_id = base_name_to_id()
        cat_to_cls = {}
        for c in cats:
            nm = str(c['name'])
            if nm not in name_to_id:
                name_to_id[nm] = len(name_to_id)
            cat_to_cls[c['id']] = name_to_id[nm]
        class_names = [None] * len(name_to_id)
        for n, i in name_to_id.items():
            class_names[i] = n

        img_info = {im['id']: im for im in coco['images']}
        per_image = {}
        for a in coco['annotations']:
            im = img_info.get(a['image_id'])
            if im is None:
                continue
            W, H = float(im['width']), float(im['height'])
            x, y, w, h = [float(v) for v in a['bbox']]
            if w < 1 or h < 1:
                continue
            cid = cat_to_cls[a['category_id']]
            line = f'{cid} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}'
            per_image.setdefault(os.path.basename(im['file_name']), []).append(line)

        if CONV_DIR.exists():
            shutil.rmtree(CONV_DIR)
        CONV_DIR.mkdir(parents=True)
        for img in images:
            lines = per_image.get(img.name)
            if lines is None:
                pairs.append((img, None))
                continue
            out = CONV_DIR / (img.stem + '.txt')
            out.write_text('\n'.join(lines), encoding='utf-8')
            pairs.append((img, out))
        print(f'변환 완료: 라벨 있는 이미지 {sum(1 for _, l in pairs if l)}장 / 전체 {len(pairs)}장')
        print('클래스 목록:', class_names)

elif images:
    print(f"[오류] LABEL_FORMAT='{LABEL_FORMAT}' 은 지원하지 않습니다. 'yolo'|'aihub_json'|'coco' 중 하나를 쓰세요.")

if pairs and class_names:
    print()
    print(f'준비 완료: 이미지 {len(pairs)}장, 클래스 {len(class_names)}개')

In [ ]:
# 3-2) 층화 분할(8:1:1) + data.yaml 자동 생성 — 그룹(개체) 인식 분할로 프레임 누수 방지
from collections import Counter, defaultdict
import re

if DATA_YAML:
    if os.path.exists(DATA_YAML):
        data_yaml_path = DATA_YAML
        print('CONFIG에 지정된 data.yaml을 그대로 사용합니다:', data_yaml_path)
        print('(층화 분할을 건너뜁니다 — split 품질은 사용자가 보장해야 합니다)')
    else:
        print('[오류] DATA_YAML 경로에 파일이 없습니다:', DATA_YAML)
elif need('pairs', 'class_names') and pairs and class_names:
    import pandas as pd
    import yaml

    random.seed(SEED)
    SPLIT_DIR = Path('/content/dataset')
    if SPLIT_DIR.exists():
        shutil.rmtree(SPLIT_DIR)
    for s in ('train', 'val', 'test'):
        (SPLIT_DIR / 'images' / s).mkdir(parents=True)
        (SPLIT_DIR / 'labels' / s).mkdir(parents=True)

    def dominant_class(lbl):
        # 이미지의 대표 클래스 = 그 이미지에서 가장 많이 등장한 클래스 (층화 기준)
        if lbl is None or not os.path.exists(lbl):
            return -1
        ids = []
        for line in open(lbl, encoding='utf-8'):
            if line.strip():
                ids.append(int(float(line.split()[0])))
        if not ids:
            return -1
        cnt = Counter(ids)
        return sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]

    def group_key(stem):
        # 같은 개체(같은 딸기 포기)를 연속 촬영한 프레임이 train/val/test에 나뉘어 들어가는
        # 데이터 누수를 막기 위한 그룹 키. (doctorgreen_aihub_data_prep.ipynb 셀 16과 동일 로직 —
        # 두 노트북의 분할 방식을 일치시켜야 하므로 로직을 바꾸려면 두 곳 모두 수정할 것)
        if GROUP_ID_REGEX:
            m = re.search(GROUP_ID_REGEX, stem)
            if m:
                return m.group(1) if m.groups() else m.group(0)
        # 자동 추정: 끝의 _0001 / -12 / 타임스탬프 같은 연속 숫자를 제거
        return re.sub(r'[_\-]?\d+$', '', stem)

    buckets = defaultdict(list)
    for img, lbl in pairs:
        buckets[dominant_class(lbl)].append((img, lbl))

    split_map = {'train': [], 'val': [], 'test': []}
    for cid in sorted(buckets):
        items = buckets[cid]
        n = len(items)

        # 개체(그룹) 인식 분할 — 그룹 키가 사실상 전부 유니크하면(그룹 수 >= 이미지 수의 95%)
        # 개체 식별이 안 되는 것으로 보고 기존 이미지 단위 랜덤 분할로 폴백한다.
        groups_map = defaultdict(list)
        for img, lbl in items:
            groups_map[group_key(img.stem)].append((img, lbl))
        gkeys = list(groups_map.keys())
        use_group = len(gkeys) < 0.95 * max(1, n)

        if use_group:
            random.shuffle(gkeys)
            n_g = len(gkeys)
            n_val_g = max(1, round(n_g * 0.1)) if n_g >= 3 else 0
            n_test_g = max(1, round(n_g * 0.1)) if n_g >= 3 else 0
            val_g = gkeys[:n_val_g]
            test_g = gkeys[n_val_g:n_val_g + n_test_g]
            train_g = gkeys[n_val_g + n_test_g:]
            val_items = [it for gk in val_g for it in groups_map[gk]]
            test_items = [it for gk in test_g for it in groups_map[gk]]
            train_items = [it for gk in train_g for it in groups_map[gk]]
            print(f'  클래스 id {cid}: 개체(그룹) {n_g}개 / 이미지 {n}장 — 그룹 단위 분할(누수 방지)')
        else:
            random.shuffle(items)
            n_val = max(1, round(n * 0.1)) if n >= 3 else 0
            n_test = max(1, round(n * 0.1)) if n >= 3 else 0
            val_items = items[:n_val]
            test_items = items[n_val:n_val + n_test]
            train_items = items[n_val + n_test:]
            print(f'  [경고] 클래스 id {cid}: 그룹 키가 대부분 유니크해(그룹 {len(gkeys)}개 / 이미지 {n}장) '
                  f'이미지 단위로 분할합니다. 연속 프레임이 있다면 GROUP_ID_REGEX 조정을 검토하세요.')

        split_map['val'] += val_items
        split_map['test'] += test_items
        split_map['train'] += train_items
        if n < 10:
            nm = class_names[cid] if 0 <= cid < len(class_names) else '(배경)'
            print(f'[주의] 클래스 "{nm}" 이미지가 {n}장뿐입니다. 통계적으로 신뢰하기 어렵습니다.')

    used = set()
    for split, items in split_map.items():
        for img, lbl in items:
            stem, k = img.stem, 1
            while stem in used:
                stem = f'{img.stem}_{k}'
                k += 1
            used.add(stem)
            shutil.copy2(img, SPLIT_DIR / 'images' / split / (stem + img.suffix.lower()))
            dst_lbl = SPLIT_DIR / 'labels' / split / (stem + '.txt')
            if lbl is not None and os.path.exists(lbl):
                shutil.copy2(lbl, dst_lbl)
            else:
                dst_lbl.write_text('', encoding='utf-8')  # 배경 이미지 = 빈 라벨

    data_yaml_path = str(SPLIT_DIR / 'data.yaml')
    with open(data_yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump({
            'path': str(SPLIT_DIR),
            'train': 'images/train',
            'val': 'images/val',
            'test': 'images/test',
            'names': {i: n for i, n in enumerate(class_names)},
        }, f, allow_unicode=True, sort_keys=False)

    # 분할 목록을 Drive에도 저장 → 어떤 파일이 어느 split이었는지 추적 가능
    split_log_dir = os.path.join(PROJECT_DIR, 'splits')
    os.makedirs(split_log_dir, exist_ok=True)
    for split, items in split_map.items():
        with open(os.path.join(split_log_dir, f'{split}.txt'), 'w', encoding='utf-8') as f:
            f.write('\n'.join(str(img) for img, _ in items))

    # 그룹(개체) 교차 검증 — train/val/test 간 같은 개체가 있으면 데이터 누수
    all_groups = {s: {group_key(img.stem) for img, _ in items} for s, items in split_map.items()}
    leak_total = 0
    print()
    print('=== 그룹(개체) 교차 검증 ===')
    for a, b in (('train', 'val'), ('train', 'test'), ('val', 'test')):
        overlap = all_groups[a] & all_groups[b]
        leak_total += len(overlap)
        print(f'  {a} ∩ {b}: {len(overlap)}건')
    if leak_total == 0:
        print('  -> train/val/test 간 그룹 교차 0건')
    else:
        print(f'  [경고] 총 {leak_total}건의 그룹 교차 발견 — 데이터 누수 가능성 있음')

    df_split = pd.DataFrame(
        [{'split': s, '이미지 수': len(v)} for s, v in split_map.items()]
    ).set_index('split')
    print()
    print(df_split)
    print()
    print('data.yaml 생성 완료:', data_yaml_path)
    print('분할 목록 저장:', split_log_dir, '(SEED 고정 → 재실행 시 동일 분할)')

## 4. 베이스라인 평가 — 현재 모델의 진짜 실력 측정

기존 배포 가중치를 방금 만든 **test split**으로 평가합니다.
개선폭(Δ)을 주장하려면 반드시 같은 test set에서의 "이전 점수"가 있어야 하므로, 이 단계는 건너뛰면 안 됩니다.

- `mAP50` : IoU 0.5 기준 — "병반을 대충이라도 찾았는가"
- `mAP50-95` : IoU 0.5~0.95 평균 — "박스가 얼마나 정확한가"
- **conf=0.75 지표** : 앱이 실제로 쓰는 컷오프에서의 정밀도/재현율 — 사용자 체감 성능

In [ ]:
metrics_base = None
if need('baseline_model', 'data_yaml_path') and baseline_model is not None:
    import pandas as pd

    # 데이터셋 클래스와 모델 클래스가 다르면 per-class 지표가 왜곡되므로 먼저 점검
    if 'class_names' in globals() and class_names and list(baseline_names.values()) != list(class_names):
        print('[주의] 배포 모델의 클래스와 data.yaml의 클래스가 다릅니다.')
        print('  모델   :', list(baseline_names.values()))
        print('  데이터 :', list(class_names))
        print('  → per-class 표는 참고용으로만 보세요.')

    try:
        metrics_base = baseline_model.val(
            data=data_yaml_path, split='test', imgsz=BASE_IMGSZ,
            plots=False, verbose=False,
        )
        print(f'베이스라인  mAP50 = {metrics_base.box.map50:.4f}   mAP50-95 = {metrics_base.box.map:.4f}')

        rows = []
        for i, ci in enumerate(metrics_base.box.ap_class_index):
            ci = int(ci)
            nm = class_names[ci] if 'class_names' in globals() and ci < len(class_names) else str(ci)
            rows.append({
                'class': nm,
                'AP50': float(metrics_base.box.ap50[i]),
                'AP50-95': float(metrics_base.box.ap[i]),
                'Precision': float(metrics_base.box.p[i]),
                'Recall': float(metrics_base.box.r[i]),
            })
        df_base = pd.DataFrame(rows)
        csv_path = os.path.join(PROJECT_DIR, 'baseline_per_class.csv')
        df_base.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print('per-class 표 저장:', csv_path)
        display(df_base.round(4))
    except Exception as e:
        print('[오류] 베이스라인 평가 실패:', e)
        print('힌트 1) 클래스 수(nc) 불일치 → 3장의 클래스 이름 처리를 확인하세요.')
        print('힌트 2) CUDA 메모리 부족 → baseline_model.val(..., batch=8)로 낮춰 보세요.')
else:
    print('베이스라인 모델이 없으면 이 셀은 건너뛰어도 됩니다. (7장 비교표에서 baseline 열이 빠질 뿐입니다)')

In [ ]:
# 앱 컷오프(conf=0.75)에서의 정밀도/재현율 — 사용자가 실제로 체감하는 성능
m075 = None
if need('baseline_model', 'data_yaml_path') and baseline_model is not None:
    try:
        m075 = baseline_model.val(
            data=data_yaml_path, split='test', imgsz=BASE_IMGSZ,
            conf=0.75, plots=False, verbose=False,
        )
        print('=== conf=0.75 (닥터그린 앱 컷오프) 기준 ===')
        print(f'  Precision = {m075.box.mp:.4f}  (진단이 떴을 때 맞을 확률)')
        print(f'  Recall    = {m075.box.mr:.4f}  (실제 병반 중 앱에 표시되는 비율)')
        print()
        print('해석: Recall이 낮으면 "병이 있는데 진단 불가"가 자주 뜬다는 뜻입니다.')
        print('      재학습으로 고신뢰 구간의 recall을 끌어올리는 것이 앱 체감 개선의 핵심입니다.')
    except Exception as e:
        print('[오류] conf=0.75 평가 실패:', e)

## 5. 데이터 감사 — mAP를 좌우하는 단계

모델 구조보다 **데이터 품질이 mAP에 훨씬 크게 기여**합니다.
클래스 불균형, 깨진 파일, 중복(누수), 병반 크기 분포를 순서대로 점검합니다.

In [ ]:
# 5-1) 클래스 분포 — 불균형이 심하면 소수 클래스의 AP가 바닥납니다
if need('data_yaml_path', 'class_names'):
    !apt-get -qq -y install fonts-nanum > /dev/null 2>&1
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm
    import numpy as np
    import pandas as pd
    from collections import Counter
    from pathlib import Path

    try:  # 그래프 한글 폰트 (실패해도 계속)
        fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
        plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        print('[안내] 한글 폰트 설치 실패 — 그래프의 한글이 깨질 수 있습니다.')
    plt.rcParams['axes.unicode_minus'] = False

    root = Path(data_yaml_path).parent
    dist = {}
    for split in ('train', 'val', 'test'):
        cnt = Counter()
        for lbl in (root / 'labels' / split).glob('*.txt'):
            for line in open(lbl, encoding='utf-8'):
                if line.strip():
                    cnt[int(float(line.split()[0]))] += 1
        dist[split] = cnt

    df_dist = pd.DataFrame(
        {s: [dist[s].get(i, 0) for i in range(len(class_names))] for s in dist},
        index=class_names,
    )
    display(df_dist)

    x = np.arange(len(class_names))
    fig, ax = plt.subplots(figsize=(max(8, len(class_names) * 1.4), 4))
    for k, split in enumerate(('train', 'val', 'test')):
        ax.bar(x + (k - 1) * 0.27, df_dist[split], width=0.27, label=split)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha='right')
    ax.set_ylabel('바운딩박스 수')
    ax.set_title('클래스 분포 (split별)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    tr = df_dist['train']
    if tr.min() > 0:
        print(f'train 불균형 비율 (최다/최소): {tr.max() / tr.min():.1f}배')
    zero = [class_names[i] for i in range(len(class_names)) if tr.iloc[i] == 0]
    if zero:
        print('[경고] train에 한 장도 없는 클래스:', zero)

**해석 가이드 — 클래스 분포**

| 이런 결과면 | 이렇게 하세요 |
|---|---|
| 최다/최소 비율 > 10배 | 소수 클래스 이미지를 추가 수집하거나, 소수 클래스 이미지를 복제(oversampling)해서 train에 추가 |
| 특정 클래스가 train에 0장 | 그 클래스는 학습 자체가 불가능 — 데이터 확보 전까지 클래스 제외 여부 결정 |
| val/test에 0장인 클래스 | 그 클래스의 AP는 측정 불가로 표시됨 — 분할 재실행(SEED 변경) 또는 데이터 추가 |

In [ ]:
# 5-2) 무결성 검사: 깨진 이미지 / 라벨 없는 이미지 / 잘못된 라벨 줄 / 중복(누수) 검사
if need('data_yaml_path'):
    import hashlib
    from pathlib import Path
    from PIL import Image
    from collections import defaultdict

    root = Path(data_yaml_path).parent
    broken, unlabeled, bad_lines = [], [], []
    hash_to_locs = defaultdict(list)

    for split in ('train', 'val', 'test'):
        for img in sorted((root / 'images' / split).glob('*')):
            try:
                with Image.open(img) as im:
                    im.verify()
            except Exception:
                broken.append(str(img))
                continue
            hash_to_locs[hashlib.md5(img.read_bytes()).hexdigest()].append((split, img.name))
            lbl = root / 'labels' / split / (img.stem + '.txt')
            txt = lbl.read_text(encoding='utf-8') if lbl.exists() else ''
            if not txt.strip():
                unlabeled.append(f'{split}/{img.name}')
            else:
                for ln, line in enumerate(txt.splitlines(), 1):
                    parts = line.split()
                    if not parts:
                        continue
                    ok = len(parts) == 5
                    if ok:
                        try:
                            vals = [float(v) for v in parts[1:]]
                            ok = all(0.0 <= v <= 1.0 for v in vals) and vals[2] > 0 and vals[3] > 0
                        except ValueError:
                            ok = False
                    if not ok:
                        bad_lines.append(f'{split}/{lbl.name}:{ln}  "{line[:60]}"')

    dups = {h: locs for h, locs in hash_to_locs.items() if len(locs) > 1}
    leaks = {h: locs for h, locs in dups.items() if len({s for s, _ in locs}) > 1}

    print(f'깨진 이미지        : {len(broken)}장')
    print(f'라벨 없는(배경) 이미지: {len(unlabeled)}장')
    print(f'형식이 잘못된 라벨 줄 : {len(bad_lines)}개')
    print(f'완전 동일(해시) 중복  : {len(dups)}묶음')
    print(f'★ split 간 중복 = 누수: {len(leaks)}묶음')
    for name, items in (('깨진 이미지', broken), ('잘못된 라벨 줄', bad_lines)):
        for it in items[:10]:
            print(f'   - [{name}] {it}')
    for h, locs in list(leaks.items())[:10]:
        print('   - [누수]', locs)

**해석 가이드 — 무결성 검사**

| 이런 결과면 | 이렇게 하세요 |
|---|---|
| 누수(split 간 중복) > 0 | **가장 심각.** 원본 데이터에서 중복 파일을 제거하고 3장을 다시 실행 — 지금까지의 mAP는 부풀려진 수치 |
| 깨진 이미지 존재 | 원본에서 삭제 후 3장 재실행 (학습 중 에러/속도 저하 원인) |
| 잘못된 라벨 줄 존재 | 해당 파일을 열어 수정 — ultralytics는 조용히 건너뛰므로 그만큼 학습 신호가 사라짐 |
| 배경(라벨 없는) 이미지 다수 | 의도된 것이면 OK(오탐 억제에 도움). 라벨 매칭 실패라면 3장의 라벨 탐색 규칙 확인 |

In [ ]:
# 5-3) 바운딩박스 크기 분포 — 작은 병반이 많으면 imgsz를 키워야 mAP가 오릅니다
if need('data_yaml_path'):
    import matplotlib.pyplot as plt
    import numpy as np
    from pathlib import Path

    root = Path(data_yaml_path).parent
    sizes = []  # imgsz=640 환산 픽셀 크기 (sqrt(면적))
    for split in ('train', 'val', 'test'):
        for lbl in (root / 'labels' / split).glob('*.txt'):
            for line in open(lbl, encoding='utf-8'):
                parts = line.split()
                if len(parts) == 5:
                    w, h = float(parts[3]), float(parts[4])
                    if w > 0 and h > 0:
                        sizes.append((w * h) ** 0.5 * 640)

    if not sizes:
        print('[오류] 라벨에서 박스를 하나도 읽지 못했습니다. 3장을 확인하세요.')
    else:
        sizes = np.array(sizes)
        small = float((sizes < 32).mean())
        medium = float(((sizes >= 32) & (sizes < 96)).mean())
        plt.figure(figsize=(8, 3.5))
        plt.hist(sizes, bins=50)
        plt.axvline(32, color='red', linestyle='--', label='32px (COCO small 기준)')
        plt.xlabel('박스 크기 (imgsz=640 환산, sqrt(면적) px)')
        plt.ylabel('박스 수')
        plt.title('바운딩박스 크기 분포')
        plt.legend()
        plt.tight_layout()
        plt.show()
        print(f'전체 박스 {len(sizes)}개 | small(<32px) {small:.1%} | medium(32~96px) {medium:.1%}')
        if small > 0.3:
            print('→ 작은 병반이 30%를 넘습니다. imgsz=896 이상을 쓰는 프리셋 B를 강력 권장합니다.')
        elif small > 0.1:
            print('→ 작은 병반이 일부 있습니다. 프리셋 A로 시작하되, 정체되면 B(imgsz=896)로 올려 보세요.')
        else:
            print('→ 병반이 대체로 큽니다. imgsz=640(프리셋 A)으로 충분할 가능성이 높습니다.')

**해석 가이드 — 박스 크기 분포**

작은 객체는 다운샘플링 과정에서 특징이 사라지기 때문에, **입력 해상도(imgsz) 상향이 작은 병반 AP를 올리는 가장 확실한 손잡이**입니다.
단, imgsz를 키우면 메모리·시간이 제곱으로 늘므로(640→896은 약 2배), 위 small 비율을 근거로 결정하세요.

In [ ]:
# 5-4) 라벨 샘플 20장 눈으로 확인 — 라벨 좌표가 틀렸으면 어떤 학습도 소용없습니다
if need('data_yaml_path', 'class_names'):
    import random as _r
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from pathlib import Path
    from PIL import Image

    root = Path(data_yaml_path).parent
    train_imgs = sorted((root / 'images' / 'train').glob('*'))
    _r.seed(SEED)
    sample = _r.sample(train_imgs, min(20, len(train_imgs)))

    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    for ax, img in zip(axes.flat, sample):
        with Image.open(img) as im:
            im = im.convert('RGB')
            W, H = im.size
            ax.imshow(im)
        lbl = root / 'labels' / 'train' / (img.stem + '.txt')
        n_box = 0
        if lbl.exists():
            for line in open(lbl, encoding='utf-8'):
                parts = line.split()
                if len(parts) != 5:
                    continue
                cid = int(float(parts[0]))
                cx, cy, w, h = [float(v) for v in parts[1:]]
                x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H
                ax.add_patch(mpatches.Rectangle((x1, y1), w * W, h * H,
                                                fill=False, edgecolor='lime', linewidth=2))
                nm = class_names[cid] if cid < len(class_names) else str(cid)
                ax.text(x1, max(0, y1 - 4), nm, color='lime', fontsize=8,
                        bbox=dict(facecolor='black', alpha=0.5, pad=1))
                n_box += 1
        ax.set_title(f'{img.name[:25]} ({n_box}개)', fontsize=8)
        ax.axis('off')
    for ax in axes.flat[len(sample):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print('체크리스트: 박스가 병반을 정확히 감싸는가? 클래스 이름이 실제 병과 일치하는가?')
    print('           라벨이 어긋나 보이면 원본 라벨 좌표계(픽셀/정규화, xywh/xyxy)를 의심하세요.')

## 6. 재학습 — 프리셋 3종

하나의 학습 셀이 `PRESET` 값에 따라 인자만 바꿔 실행됩니다. COCO 사전학습 가중치에서 전이학습합니다.

| 프리셋 | 베이스 | 핵심 설정 | 예상 시간(L4, 3천장 기준) | 언제 쓰나 |
|---|---|---|---|---|
| **A** 빠른 개선 | `yolov8s.pt` | imgsz=640, epochs=100, patience=20, close_mosaic=10 | ~1시간 | 첫 실험, 파이프라인 검증 |
| **B** 정확도 우선 | `yolo11m.pt` | imgsz=896, epochs=150, cos_lr, mixup=0.1, copy_paste=0.1 | 2~4시간 | 작은 병반 다수, 최종 제출용 |
| **C** 탐색 | `yolo11m.pt` | `model.tune()` 20회×30ep 후 최적값으로 본 학습 | 반나절 | A/B가 정체됐을 때 |

- `batch=-1`(자동 배치)과 `cache=True`(RAM 캐시)로 GPU를 최대한 활용합니다.
- 결과는 `PROJECT_DIR/train_<PRESET>/`(Drive)에 저장됩니다.

> **세션이 끊겼을 때 이어서 학습(resume)**
> 1. 0~1장(CONFIG, 환경)과 3장(데이터 준비) 셀을 다시 실행 (SEED 고정 → 동일 분할 재현)
> 2. 새 셀에서 아래 두 줄 실행:
> ```python
> from ultralytics import YOLO
> YOLO(f'{PROJECT_DIR}/train_A/weights/last.pt').train(resume=True)
> ```

In [ ]:
best_pt = None
if need('data_yaml_path'):
    import glob
    import yaml
    from ultralytics import YOLO

    PRESETS = {
        'A': {'base': 'yolov8s.pt',
              'args': dict(imgsz=640, epochs=100, patience=20, close_mosaic=10,
                           batch=-1, cache=True)},
        'B': {'base': 'yolo11m.pt',
              'args': dict(imgsz=896, epochs=150, patience=30, batch=-1, cache=True,
                           cos_lr=True, mixup=0.1, copy_paste=0.1)},
        'C': {'base': 'yolo11m.pt',
              'args': dict(imgsz=896, epochs=150, patience=30, batch=-1, cache=True,
                           cos_lr=True)},
    }
    if PRESET not in PRESETS:
        print(f"[오류] PRESET='{PRESET}' 은 'A'|'B'|'C' 중 하나여야 합니다.")
    else:
        preset = PRESETS[PRESET]
        run_name = 'train_' + PRESET
        best_hyp = {}

        try:
            if PRESET == 'C':
                print('[C] 하이퍼파라미터 탐색 시작 (iterations=20 × epochs=30, 수 시간 소요)...')
                tuner = YOLO(preset['base'])
                tuner.tune(data=data_yaml_path, epochs=30, iterations=20,
                           imgsz=preset['args']['imgsz'], batch=16,
                           plots=False, save=False, val=True,
                           project=PROJECT_DIR, name='tune_C', exist_ok=True)
                cands = sorted(
                    glob.glob(os.path.join(PROJECT_DIR, '**', 'best_hyperparameters.yaml'), recursive=True)
                    + glob.glob('runs/**/best_hyperparameters.yaml', recursive=True),
                    key=os.path.getmtime)
                if cands:
                    best_hyp = yaml.safe_load(open(cands[-1], encoding='utf-8')) or {}
                    print('탐색된 최적 하이퍼파라미터:')
                    for k, v in best_hyp.items():
                        print(f'  {k} = {v}')
                else:
                    print('[경고] best_hyperparameters.yaml을 찾지 못해 기본값으로 본 학습을 진행합니다.')

            model = YOLO(preset['base'])
            train_kwargs = {**preset['args'], **best_hyp}
            results = model.train(data=data_yaml_path, project=PROJECT_DIR, name=run_name,
                                  exist_ok=True, seed=SEED, plots=True, **train_kwargs)

            best_pt = str(getattr(model.trainer, 'best', '') or '')
            if not best_pt or not os.path.exists(best_pt):
                best_pt = os.path.join(PROJECT_DIR, run_name, 'weights', 'best.pt')
            print()
            print('학습 완료. best.pt (Drive에 보존됨):', best_pt)
        except Exception as e:
            print('[오류] 학습 실패:', e)
            print('힌트 1) CUDA out of memory → args에 batch=8(또는 4) 고정, cache=False, imgsz 축소')
            print('힌트 2) 세션이 끊긴 경우 → 위 마크다운의 resume 안내대로 이어서 학습하세요.')
            print("힌트 3) 'No labels found' → 3장에서 라벨 변환/분할이 제대로 됐는지 확인하세요.")

In [ ]:
# 학습 곡선 확인 — loss는 내려가는데 val mAP가 정체/하락하면 과적합 신호입니다
if need('data_yaml_path'):
    from IPython.display import Image as IPyImage, display

    run_dir = os.path.join(PROJECT_DIR, 'train_' + PRESET)
    png = os.path.join(run_dir, 'results.png')
    csv = os.path.join(run_dir, 'results.csv')

    if os.path.exists(png):
        display(IPyImage(filename=png, width=1000))
    else:
        print('results.png가 아직 없습니다. 학습이 정상 종료됐는지 확인하세요:', run_dir)

    if os.path.exists(csv):
        import pandas as pd
        df_res = pd.read_csv(csv)
        df_res.columns = df_res.columns.str.strip()
        col = 'metrics/mAP50-95(B)'
        if col in df_res.columns:
            bi = int(df_res[col].idxmax())
            print(f'best epoch = {int(df_res.loc[bi, "epoch"])}  '
                  f'mAP50={df_res.loc[bi, "metrics/mAP50(B)"]:.4f}  '
                  f'mAP50-95={df_res.loc[bi, col]:.4f}')
            if bi >= len(df_res) - 3:
                print('→ 마지막까지 계속 오르고 있었습니다. epochs를 늘리면 더 오를 여지가 있습니다.')

## 7. 비교 평가 — 새 모델 vs 베이스라인

**같은 test split**에서 두 모델을 평가해 개선폭(Δ)을 확인합니다.
per-class Δ가 큰 폭으로 하락한 클래스가 있다면, 전체 mAP가 올라도 배포를 재고해야 합니다.

In [ ]:
metrics_new = None
new_model = None
if need('best_pt', 'data_yaml_path', 'class_names') and best_pt and os.path.exists(best_pt):
    import pandas as pd
    from ultralytics import YOLO

    mb = globals().get('metrics_base')  # 4장을 건너뛰었어도 동작하도록
    try:
        new_model = YOLO(best_pt)
        eval_imgsz = preset['args']['imgsz'] if 'preset' in globals() else 640
        metrics_new = new_model.val(data=data_yaml_path, split='test',
                                    imgsz=eval_imgsz, plots=True, verbose=False)

        def per_class_df(metrics, tag):
            rows = []
            for i, ci in enumerate(metrics.box.ap_class_index):
                ci = int(ci)
                nm = class_names[ci] if ci < len(class_names) else str(ci)
                rows.append({'class': nm,
                             f'AP50_{tag}': float(metrics.box.ap50[i]),
                             f'AP50-95_{tag}': float(metrics.box.ap[i])})
            return pd.DataFrame(rows)

        print('=== 전체 지표 ===')
        if mb is not None:
            print(f'  mAP50    : {mb.box.map50:.4f} → {metrics_new.box.map50:.4f}  '
                  f'(Δ {metrics_new.box.map50 - mb.box.map50:+.4f})')
            print(f'  mAP50-95 : {mb.box.map:.4f} → {metrics_new.box.map:.4f}  '
                  f'(Δ {metrics_new.box.map - mb.box.map:+.4f})')
            df_cmp = per_class_df(mb, 'base').merge(
                per_class_df(metrics_new, 'new'), on='class', how='outer')
            df_cmp['ΔAP50'] = df_cmp['AP50_new'] - df_cmp['AP50_base']
            df_cmp['ΔAP50-95'] = df_cmp['AP50-95_new'] - df_cmp['AP50-95_base']
        else:
            print(f'  (베이스라인 없음)  mAP50 = {metrics_new.box.map50:.4f}   '
                  f'mAP50-95 = {metrics_new.box.map:.4f}')
            df_cmp = per_class_df(metrics_new, 'new')

        csv_path = os.path.join(PROJECT_DIR, 'compare_per_class.csv')
        df_cmp.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print('비교 표 저장:', csv_path)
        display(df_cmp.round(4))
        if 'ΔAP50' in df_cmp.columns:
            worse = df_cmp[df_cmp['ΔAP50'] < -0.02]['class'].tolist()
            if worse:
                print('[주의] AP50이 하락한 클래스:', worse, '→ 8장 오류 분석에서 원인을 확인하세요.')
    except Exception as e:
        print('[오류] 비교 평가 실패:', e)
        print('힌트) best_pt 경로 확인:', best_pt)
else:
    print('best.pt가 없습니다. 6장 학습을 먼저 완료하세요.')
    print('이전 세션에서 학습을 마쳤다면: best_pt = f"{PROJECT_DIR}/train_A/weights/best.pt" 처럼 직접 지정 후 재실행하세요.')

In [ ]:
# confusion matrix + PR curve — 어떤 클래스끼리 헷갈리는지, 어느 conf 구간이 약한지
if need('metrics_new') and metrics_new is not None:
    from IPython.display import Image as IPyImage, display

    save_dir = str(metrics_new.save_dir)
    print('평가 산출물 폴더:', save_dir)
    for fname, desc in [
        ('confusion_matrix_normalized.png', '혼동 행렬(정규화) — 대각선 밖 큰 값 = 서로 헷갈리는 클래스 쌍'),
        ('BoxPR_curve.png', 'PR 곡선 — 오른쪽 위로 붙을수록 좋음'),
        ('BoxF1_curve.png', 'F1-conf 곡선 — 앱 컷오프 0.75 지점의 F1을 확인하세요'),
    ]:
        p = os.path.join(save_dir, fname)
        if os.path.exists(p):
            print()
            print('▶', desc)
            display(IPyImage(filename=p, width=800))
        else:
            print(f'({fname} 없음)')

In [ ]:
# conf=0.75(앱 체감) 재측정 + TTA(테스트 시 증강) 효과
if need('new_model', 'data_yaml_path') and new_model is not None:
    eval_imgsz = preset['args']['imgsz'] if 'preset' in globals() else 640
    try:
        m_new075 = new_model.val(data=data_yaml_path, split='test', imgsz=eval_imgsz,
                                 conf=0.75, plots=False, verbose=False)
        mb075 = globals().get('m075')  # 4장을 건너뛰었어도 동작하도록
        print('=== conf=0.75 (앱 컷오프) 비교 ===')
        if mb075 is not None:
            print(f'  Precision : {mb075.box.mp:.4f} → {m_new075.box.mp:.4f}')
            print(f'  Recall    : {mb075.box.mr:.4f} → {m_new075.box.mr:.4f}')
        else:
            print(f'  Precision = {m_new075.box.mp:.4f}   Recall = {m_new075.box.mr:.4f}')

        print()
        print('=== TTA(augment=True) 효과 — 추론이 2~3배 느려지는 대신 mAP 소폭 상승 ===')
        m_tta = new_model.val(data=data_yaml_path, split='test', imgsz=eval_imgsz,
                              augment=True, plots=False, verbose=False)
        print(f'  mAP50    : {metrics_new.box.map50:.4f} → {m_tta.box.map50:.4f} (TTA)')
        print(f'  mAP50-95 : {metrics_new.box.map:.4f} → {m_tta.box.map:.4f} (TTA)')
        print('  주의: Space는 실시간 서비스이므로 TTA 상시 적용은 응답 지연을 유발합니다.')
        print('        수치 참고용으로만 보고, 배포는 TTA 없는 기본 추론 기준으로 판단하세요.')
    except Exception as e:
        print('[오류] 측정 실패:', e)

## 8. 오류 분석 — 다음 라운드에 어떤 데이터를 보강할까

val 예측과 정답을 비교해 **놓친 병반(FN)** 과 **오탐(FP)** 이 많은 이미지를 시각화합니다.
mAP를 더 올리는 가장 빠른 길은 "모델이 틀리는 패턴의 데이터"를 집중 보강하는 것입니다.

- **FN이 많다** → 그 조건(역광·초기 병반·잎 뒷면 등)의 이미지를 추가 수집/라벨링
- **FP가 많다** → 헷갈리는 배경(물방울·흙·오래된 상처 등)이 담긴 **배경 이미지(빈 라벨)** 를 train에 추가
- 특정 클래스 쌍에서 오분류가 반복 → 라벨 기준 자체가 애매하지 않은지 라벨 가이드 재점검

초록 실선 = 정답(GT), 빨간 점선 = 모델 예측입니다.

In [ ]:
if need('new_model', 'data_yaml_path') and new_model is not None:
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from pathlib import Path
    from PIL import Image

    def box_iou_np(a, b):
        if len(a) == 0 or len(b) == 0:
            return np.zeros((len(a), len(b)))
        iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
        ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
        inter = iw * ih
        area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
        area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
        return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

    root = Path(data_yaml_path).parent
    val_imgs = sorted(str(p) for p in (root / 'images' / 'val').glob('*'))
    records = []
    try:
        for r in new_model.predict(val_imgs, conf=0.25, verbose=False, stream=True):
            H, W = r.orig_shape
            lbl = root / 'labels' / 'val' / (Path(r.path).stem + '.txt')
            gt_boxes, gt_cls = [], []
            if lbl.exists():
                for line in open(lbl, encoding='utf-8'):
                    parts = line.split()
                    if len(parts) == 5:
                        cid = int(float(parts[0]))
                        cx, cy, w, h = [float(v) for v in parts[1:]]
                        gt_boxes.append([(cx - w / 2) * W, (cy - h / 2) * H,
                                         (cx + w / 2) * W, (cy + h / 2) * H])
                        gt_cls.append(cid)
            gt_boxes = np.array(gt_boxes).reshape(-1, 4)
            gt_cls = np.array(gt_cls, dtype=int)
            pb = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.zeros((0, 4))
            pc = r.boxes.cls.cpu().numpy().astype(int) if r.boxes is not None else np.zeros(0, dtype=int)
            pconf = r.boxes.conf.cpu().numpy() if r.boxes is not None else np.zeros(0)

            # conf 내림차순 greedy 매칭 (IoU>=0.5, 같은 클래스)
            iou = box_iou_np(pb, gt_boxes)
            gt_used = np.zeros(len(gt_boxes), dtype=bool)
            pred_tp = np.zeros(len(pb), dtype=bool)
            for pi in np.argsort(-pconf):
                cand = [gi for gi in range(len(gt_boxes))
                        if not gt_used[gi] and gt_cls[gi] == pc[pi] and iou[pi, gi] >= 0.5]
                if cand:
                    gi = max(cand, key=lambda g: iou[pi, g])
                    gt_used[gi] = True
                    pred_tp[pi] = True
            records.append({'path': r.path, 'fn': int((~gt_used).sum()), 'fp': int((~pred_tp).sum()),
                            'gt': gt_boxes, 'gt_cls': gt_cls, 'pb': pb, 'pc': pc, 'pconf': pconf})

        def show_grid(recs, title):
            if not recs:
                print(f'{title}: 해당 이미지 없음')
                return
            fig, axes = plt.subplots(2, 4, figsize=(22, 11))
            fig.suptitle(title, fontsize=14)
            for ax, rec in zip(axes.flat, recs):
                with Image.open(rec['path']) as im:
                    ax.imshow(im.convert('RGB'))
                for b, c in zip(rec['gt'], rec['gt_cls']):
                    ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1],
                                                    fill=False, edgecolor='lime', linewidth=2))
                    nm = class_names[c] if c < len(class_names) else str(c)
                    ax.text(b[0], max(0, b[1] - 4), nm, color='lime', fontsize=7,
                            bbox=dict(facecolor='black', alpha=0.5, pad=1))
                for b, c, cf in zip(rec['pb'], rec['pc'], rec['pconf']):
                    ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1],
                                                    fill=False, edgecolor='red', linewidth=1.5, linestyle='--'))
                    nm = class_names[c] if c < len(class_names) else str(c)
                    ax.text(b[0], b[3] + 4, f'{nm} {cf:.2f}', color='red', fontsize=7,
                            bbox=dict(facecolor='black', alpha=0.5, pad=1))
                ax.set_title(f"{Path(rec['path']).name[:28]}  FN={rec['fn']} FP={rec['fp']}", fontsize=8)
                ax.axis('off')
            for ax in axes.flat[len(recs):]:
                ax.axis('off')
            plt.tight_layout()
            plt.show()

        show_grid(sorted(records, key=lambda x: -x['fn'])[:8], 'FN 상위 8장 — 모델이 놓친 병반 (초록=정답, 빨강 점선=예측)')
        show_grid(sorted(records, key=lambda x: -x['fp'])[:8], 'FP 상위 8장 — 모델의 오탐')
        print(f'val {len(records)}장 기준  총 FN={sum(r["fn"] for r in records)}, 총 FP={sum(r["fp"] for r in records)} (conf>=0.25, IoU>=0.5)')
    except Exception as e:
        print('[오류] 오류 분석 실패:', e)
        print('힌트) val 이미지 폴더 확인:', str(root / 'images' / 'val'))

## 9. 배포 — Hugging Face Space에 새 가중치 올리기

> **[주의] 업로드 전 필수 확인 — 앱이 깨질 수 있습니다**
>
> Space의 Gradio 서버(app.py)는 `model.names`의 **클래스 이름과 인덱스 순서**를 기준으로
> `disease_name`, `detections[].name` 등 응답 스키마를 만들어 닥터그린 앱에 보냅니다.
> 클래스 이름·순서가 조금이라도 바뀌면 앱의 진단 결과 화면이 잘못된 병명을 표시하거나 깨집니다.
> **반드시 바로 아래 검증 셀을 실행해 "일치"를 확인한 뒤** 업로드하세요.
> 불일치 상태로 배포해야 한다면 Space의 app.py 매핑 코드도 함께 수정해야 합니다.

**방법 (a) — 웹 UI로 교체 (간단, 권장)**
1. `PROJECT_DIR/train_<PRESET>/weights/best.pt`를 Drive에서 PC로 내려받기
2. `https://huggingface.co/spaces/henna22/doctor-green-strawberry/tree/main` 접속
3. 기존 `.pt` 파일과 **같은 경로·같은 파일명**으로 업로드(교체) → 커밋하면 Space가 자동 재빌드

**방법 (b) — 아래 코드 셀로 업로드 (huggingface_hub)**
- write 권한 토큰으로 `notebook_login()` 후, 검증 통과 시 `DO_UPLOAD = True`로 바꿔 실행

**업로드 후 확인 절차**
1. Space가 재시작될 때까지 1~2분 대기 (Space 페이지에서 Running 상태 확인)
2. 닥터그린 앱의 `/api/diagnose/ping`을 호출해 **워밍업**(콜드 스타트 해소)
3. 실제 병반 사진 몇 장으로 진단 테스트 — 앱 컷오프가 0.75이므로 conf 0.75 이상으로 잡히는지 확인

In [ ]:
# 9-1) 클래스 이름/순서 검증 — 통과하지 못하면 업로드 금지
names_match = False
if need('best_pt') and best_pt and os.path.exists(best_pt):
    import pandas as pd
    from ultralytics import YOLO

    new_names = dict(YOLO(best_pt).names)
    if 'baseline_names' in globals() and baseline_names:
        idxs = sorted(set(baseline_names) | set(new_names))
        df_names = pd.DataFrame([{
            'index': i,
            '기존(Space)': baseline_names.get(i, '(없음)'),
            '신규(best.pt)': new_names.get(i, '(없음)'),
            '일치': baseline_names.get(i) == new_names.get(i),
        } for i in idxs])
        display(df_names)
        names_match = bool(df_names['일치'].all())
        if names_match:
            print('[통과] 클래스 이름/순서 일치 — 업로드해도 앱 스키마가 유지됩니다.')
        else:
            print('[경고] 클래스 이름/순서 불일치!')
            print('   이대로 업로드하면 Space app.py의 disease_name 매핑과 앱 응답 스키마가 깨집니다.')
            print('   조치: 3장의 클래스 이름 처리(배포 모델 순서 재사용)를 확인하거나, app.py를 함께 수정하세요.')
    else:
        print('[안내] 베이스라인 클래스 목록이 없어 자동 비교를 못 했습니다.')
        print('       새 모델 클래스:', new_names)
        print('       Space app.py가 기대하는 이름/순서와 직접 대조한 뒤 진행하세요.')
else:
    print('best.pt가 없습니다. 6장 학습을 먼저 완료하세요.')

In [ ]:
# 9-2) huggingface_hub로 업로드 — 검증 통과 후 DO_UPLOAD를 True로 바꿔 실행
DO_UPLOAD = False   # ← 9-1 검증 통과를 확인한 뒤에만 True로 변경

if need('best_pt') and best_pt and os.path.exists(best_pt):
    if not DO_UPLOAD:
        print('DO_UPLOAD = False 상태입니다. (실수 방지용 안전장치)')
        print('9-1 검증에서 "일치"를 확인했다면 이 셀 첫 줄을 DO_UPLOAD = True로 바꿔 다시 실행하세요.')
        print('업로드 대상 파일:', best_pt)
    elif not globals().get('names_match', False):
        print('[중단] 9-1 클래스 검증을 통과하지 못했거나 실행하지 않았습니다. 9-1 셀을 먼저 실행하세요.')
        print('       (app.py를 함께 수정하는 계획이라면 이 조건문을 직접 우회하세요 — 권장하지 않음)')
    else:
        from huggingface_hub import upload_file
        # 최초 1회 로그인(write 토큰): from huggingface_hub import notebook_login; notebook_login()
        target = space_pt_relpath if ('space_pt_relpath' in globals() and space_pt_relpath) else 'best.pt'
        try:
            url = upload_file(
                path_or_fileobj=best_pt,
                path_in_repo=target,
                repo_id=HF_SPACE,
                repo_type='space',
                commit_message=f'update: mAP-boosted weights (preset {PRESET}, seed {SEED})',
            )
            print('[완료] 업로드 성공:', url)
            print()
            print('다음 단계:')
            print('  1) Space가 자동 재시작됩니다 — 1~2분 뒤 Running 상태 확인')
            print('  2) 닥터그린 앱의 /api/diagnose/ping 호출로 워밍업')
            print('  3) 실제 병반 사진으로 진단 테스트 (앱 컷오프 conf=0.75 기준)')
        except Exception as e:
            print('[오류] 업로드 실패:', e)
            print('힌트 1) 401/403 → write 권한 토큰으로 notebook_login()을 먼저 실행하세요.')
            print('힌트 2) repo를 찾을 수 없음 → HF_SPACE 철자를 확인하세요:', HF_SPACE)